# Module 10: Numerical Methods & Computational Math

Computers represent numbers using finite-precision arithmetic, leading to rounding errors, underflow, and overflow. Numerical methods study how to design algorithms that are computationally stable, efficient, and accurate. This is crucial for deep learning, where training algorithms run billions of calculations.

## Contents
1. Floating-Point Representation and Precision Issues
2. Numerical Stability (Overflow, Underflow, Log-Sum-Exp)
3. Condition Numbers and System Conditioning
4. Iterative Methods for Linear Systems (Conjugate Gradient)
5. Numerical Integration & Monte Carlo Methods
6. Pseudorandom Number Generation

## 1. Floating-Point Representation and Precision Issues

Computers represent real numbers using the IEEE 754 floating-point standard. A number is represented as:
$$x = (-1)^s \times m \times 2^{e - bias}$$
where $s$ is the sign bit, $m$ is the mantissa (significand), and $e$ is the exponent.

### Precision Formats in Deep Learning:
- **FP32** (Single precision): 1 sign bit, 8 exponent bits, 23 mantissa bits.
- **FP16** (Half precision): 1 sign bit, 5 exponent bits, 10 mantissa bits. Faster computation but susceptible to overflow/underflow.
- **BF16** (Brain Floating Point): 1 sign bit, 8 exponent bits, 7 mantissa bits. Same range as FP32, making it easier to train networks without gradient scaling.
- **INT8/FP8**: Used heavily in quantization for inference acceleration.

Let's see floating point precision limits (machine epsilon) in Python.

In [ ]:
import numpy as np

# Machine epsilon for float32 and float64
print("Float32 Machine Epsilon:", np.finfo(np.float32).eps)
print("Float64 Machine Epsilon:", np.finfo(np.float64).eps)

# Demonstration of loss of significance
a = 1e16
b = 1.0
print(f"a + b - a = {a + b - a} (Expected: 1.0)")

## 2. Numerical Stability: Log-Sum-Exp Trick

When computing Softmax or cross-entropy, we evaluate terms like $e^{x_i}$. If $x_i$ is large, $e^{x_i}$ overflows to `inf`. If $x_i$ is highly negative, it underflows to `0`.

To compute $\log \sum_i e^{x_i}$ stably, we use the **Log-Sum-Exp (LSE) Trick**:
$$\log \sum_{i=1}^n e^{x_i} = c + \log \sum_{i=1}^n e^{x_i - c}$$
where we typically choose $c = \max_i x_i$. This guarantees that the largest exponent is $0$ ($e^0 = 1$), completely preventing overflow and reducing underflow.

In [ ]:
def unstable_lse(x):
    return np.log(np.sum(np.exp(x)))

def stable_lse(x):
    c = np.max(x)
    return c + np.log(np.sum(np.exp(x - c)))

x = np.array([1000.0, 1001.0, 1002.0])
try:
    print("Unstable:", unstable_lse(x))
except Exception as e:
    print("Unstable failed!")
print("Stable:", stable_lse(x))

## 3. Condition Numbers and System Conditioning

The **condition number** $\kappa(A)$ of a matrix measures how sensitive the output of $A\mathbf{x} = \mathbf{b}$ is to small perturbations in input data or round-off errors.
$$\kappa(A) = \|A\| \|A^{-1}\|$$

For the L2 norm, the condition number of a normal matrix is the ratio of its largest to smallest eigenvalue (in absolute value):
$$\kappa(A) = \frac{|\lambda_{\max}(A)|}{|\lambda_{\min}(A)|}$$

- If $\kappa(A) \approx 1$, the system is **well-conditioned**.
- If $\kappa(A) \gg 1$, the system is **ill-conditioned**, meaning small errors in $\mathbf{b}$ lead to huge errors in $\mathbf{x}$.

In [ ]:
A = np.array([[1.0, 1.0], [1.0, 1.0001]])
print("Condition number:", np.linalg.cond(A))

b1 = np.array([2.0, 2.0])
b2 = np.array([2.0, 2.0001])

x1 = np.linalg.solve(A, b1)
x2 = np.linalg.solve(A, b2)
print("x1:", x1)
print("x2:", x2)
print("Absolute difference:", np.linalg.norm(x1 - x2))

## 4. Iterative Methods for Linear Systems (Conjugate Gradient)

For huge, sparse systems, direct methods like LU factorization ($O(N^3)$) are too slow. We use iterative solvers instead.

**Conjugate Gradient (CG)** is a highly effective method to solve $A\mathbf{x} = \mathbf{b}$ for symmetric positive-definite $A$. Instead of moving in orthogonal directions, it searches along $A$-conjugate directions:
$$\mathbf{p}_i^T A \mathbf{p}_j = 0 \quad \forall i \ne j$$

This guarantees convergence to the exact solution in at most $N$ steps (in exact arithmetic).

In [ ]:
from scipy.sparse.linalg import cg

A = np.array([[4.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, 2.0])

x_sol, exit_code = cg(A, b)
print("CG Solution:", x_sol)

## 5. Numerical Integration & Monte Carlo Methods

When an integral is analytically intractable (e.g., in Bayesian posterior calculations or diffusion models), we approximate it.

### Monte Carlo Integration
To compute $I = \int_a^b f(x) dx$, we write it as an expectation:
$$I = (b - a) \mathbb{E}[f(X)] \approx \frac{b-a}{N} \sum_{i=1}^N f(x_i)$$
where $x_i \sim \text{Uniform}(a, b)$. By the Central Limit Theorem, the error decreases at a rate of $O(1/\sqrt{N})$, which is independent of the dimensionality of the integration domain. This makes Monte Carlo the only viable method for high-dimensional integration in machine learning.

In [ ]:
# Estimate Pi using Monte Carlo integration (area of circle)
N = 100000
x = np.random.uniform(-1, 1, N)
y = np.random.uniform(-1, 1, N)
inside = x**2 + y**2 <= 1.0
pi_est = 4 * np.sum(inside) / N
print("Estimated Pi:", pi_est)

## 6. Pseudorandom Number Generation

Computers cannot generate truly random numbers; they use deterministic algorithms called **Pseudorandom Number Generators (PRNGs)**. 
- **Linear Congruential Generator (LCG)**: $X_{n+1} = (a X_n + c) \pmod m$.
- **Mersenne Twister**: Standard generator used by Python (`random` and `numpy.random`). Highly performant, period of $2^{19937}-1$.
- **JAX/PRNG key system**: Uses cryptographic hashing functions to generate random states explicitly. Essential for reproducibility in parallel code.